In [8]:
import pytesseract

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

In [2]:
import matplotlib.pyplot as plt
import cv2


In [9]:
print(pytesseract.get_languages(config=''))

['eng', 'osd']


In [ ]:
import cv2
import pytesseract
from ultralytics import YOLO
from collections import deque, Counter

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

model = YOLO("license_plate_detector.pt")

cap = cv2.VideoCapture("videoplayback.mp4")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter(
    "output_result.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (width, height)
)

buffer = deque(maxlen=10)

def clean_text(text):
    return text.strip().replace(" ", "")

def get_stable_text():
    if len(buffer) == 0:
        return ""
    return Counter(buffer).most_common(1)[0][0]

while True:
    ret, frame = cap.read()
    if not ret or frame is None:
        break

    results = model.predict(frame, verbose=False)

    for r in results:
        if r.boxes is None:
            continue

        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            h, w = frame.shape[:2]
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w, x2), min(h, y2)

            plate = frame[y1:y2, x1:x2]

            if plate.size == 0:
                continue

            gray = cv2.cvtColor(plate, cv2.COLOR_BGR2GRAY)
            gray = cv2.resize(gray, None, fx=2, fy=2)
            gray = cv2.GaussianBlur(gray, (5,5), 0)
            _, thresh = cv2.threshold(gray, 140, 255, cv2.THRESH_BINARY)

            text = pytesseract.image_to_string(thresh, config='--oem 3 --psm 7')
            text = clean_text(text)

            if len(text) >= 5:
                buffer.append(text)

            stable_text = get_stable_text()

            cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
            cv2.putText(frame, stable_text, (x1, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)

    cv2.imshow("Stable Plate Recognition", frame)

    out.write(frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
out.release()
cv2.destroyAllWindows()

In [37]:
import cv2
import pytesseract
from ultralytics import YOLO
from collections import Counter

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

model = YOLO("license_plate_detector.pt")

cap = cv2.VideoCapture("videoplayback.mp4")

all_plates = [] 

def clean_text(text):
    return text.strip().replace(" ", "")

while True:
    ret, frame = cap.read()
    if not ret or frame is None:
        break

    results = model.predict(frame, verbose=False)

    for r in results:
        if r.boxes is None:
            continue

        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            h, w = frame.shape[:2]
            x1, y1 = max(0,x1), max(0,y1)
            x2, y2 = min(w,x2), min(h,y2)

            plate = frame[y1:y2, x1:x2]

            if plate.size == 0:
                continue

            gray = cv2.cvtColor(plate, cv2.COLOR_BGR2GRAY)
            gray = cv2.resize(gray, None, fx=2, fy=2)
            _, thresh = cv2.threshold(gray, 140, 255, cv2.THRESH_BINARY)

            text = pytesseract.image_to_string(thresh, config='--oem 3 --psm 7')
            text = clean_text(text)

            if len(text) >= 4:
                all_plates.append(text)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()

if len(all_plates) > 0:
    final_plate = Counter(all_plates).most_common(1)[0][0]
else:
    final_plate = "NOT FOUND"

print("🏁 Most Frequent Plate:", final_plate)

with open("final_plate.txt", "w") as f:
    f.write(final_plate)

🏁 Most Frequent Plate: *6J38BC9025
